# DAT09 · 미션8 대량 입력과 저장 실패 시험하기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 3.5시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09_3.md`(또는 `DAT09.md`)를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션7을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m07.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m08_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09_3.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09_3.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=7):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09_3.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 7)
else:
    print('DAT09_3.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09_3.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션8 대량 입력과 저장 실패 시험하기

예상 3.5시간입니다.

#### 이번 미션의 목표

대량 입력을 실제 변환·검증·DB 저장까지 처리하고, 실패한 보고도 빠짐없이 기록하며 성능과 업무시간 가정을 구별합니다.

### 1단계 세 규모의 실제 부하 시험 입력 만들기

#### 이 단계에서 하는 일

작은 파일의 실행시간을 곱해서 대량 처리 성능이라고 할 수 없습니다. 실제 크기의 입력을 만들고 정상·오류·재전송 건수를 알고 시험해야 처리 누락을 찾을 수 있습니다.

#### 시작 전 확인

미션7까지 완료한 결과를 사용합니다. 노트북의 m08_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

generate_data.py의 create_load와 config/load_profile.json을 사용합니다. 여기서 원본 건수는 서로 다른 이벤트 수이며, 재전송은 같은 이벤트를 한 번 더 보내는 입력입니다.

#### 작업 순서

- 고유 이벤트 5,000·20,000·50,000건을 각각 생성하고 각 규모의 1%인 50·200·500건을 재전송으로 더하세요. 실제 파일의 줄 수를 세어 확인합니다.

- 고유 이벤트의 0.4%인 20·80·200건에는 수량 형식 오류를 넣으세요. 같은 오류 이벤트의 재전송이 있다면 오류 수신 건수와 고유 오류 수는 다를 수 있습니다.

- 생성 시드와 입력 조건, 파일 경로·해시·고유 건수·전체 수신 건수·고유 오류 수를 저장하세요. 생성한 파일을 다음 단계에서 실제로 읽습니다.


#### 저장할 결과

data/load_5000.jsonl, load_20000.jsonl, load_50000.jsonl과 각 manifest, outputs/M08/load_manifest.json을 저장합니다.

#### 직접 확인하는 방법

전체 수신은 5,050·20,200·50,500건입니다. 고유 오류는 20·80·200건이고 재전송과 구별해야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 기존 처리 함수를 큐와 DB 저장에 연결하기

#### 이 단계에서 하는 일

문자열을 읽는 속도만 재면 실제 업무 처리 성능을 알 수 없습니다. 앞서 만든 변환·검증·이벤트 처리를 거쳐 DB에 확정 저장하고, 각 수신의 상태를 남기는 범위까지 측정합니다.

#### 시작 전 확인

미션8의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

세 부하 입력, 미션2 normalize_fn·status_fn, 미션3 rule_fn, 미션4 이벤트 처리 기준과 SQLite를 사용합니다. 큐는 처리 전 보고가 기다리는 목록이며 이 실습은 Python 스레드로 유입을 모의합니다.

#### 작업 순서

- A·B·C 출처별 생산자 스레드 세 개가 이벤트를 큐에 넣고 소비자 한 개가 꺼내게 하세요. 생산자 종료 신호를 모두 받은 뒤 남은 보고까지 처리합니다.

- 500건씩 묶어 변환·검증·중복 ID·버전 처리를 수행하세요. DB에는 이미 본 ID, 대상별 현재 상태, 수신별 상태 원장을 저장합니다. 500건 미만인 마지막 묶음도 빠뜨리지 않습니다.

- 각 실행마다 새 시험 DB를 만들고 한 묶음의 저장을 트랜잭션으로 처리하세요. 성공한 commit 뒤에만 저장 완료로 기록합니다.

- 파일 읽기부터 변환·검증·DB 저장·상태 기록까지 perf_counter로 시간을 재세요. 독립 검산 시간은 별도이며 측정 범위에 넣었는지 명시합니다. 5천 규모로 워밍업 한 번을 실행합니다.


#### 저장할 결과

실제 처리 함수 c['process_load_fn'], 워밍업 결과, 실행별 SQLite DB·event_status.csv·queue_log.json·retry_log.json을 남깁니다.

#### 직접 확인하는 방법

큐가 비어 보인다는 이유만으로 생산자가 끝나기 전에 종료하면 안 됩니다. 워밍업도 입력 수와 상태 원장 합계가 같고 대상별 검산 차이가 0인지 확인합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 저장 실패 후 재시도와 최종 실패 기록 확인하기

#### 이 단계에서 하는 일

저장이 실패했는데 메모리에서만 처리 완료로 남으면 재시도 때 보고가 누락될 수 있습니다. 반대로 일부 저장을 남기면 재시도 때 중복될 수 있습니다. 실패한 묶음을 되돌린 뒤 다시 시도하는지 확인합니다.

#### 시작 전 확인

미션8의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

5,050건 입력과 process_load_fn을 사용합니다. 정상 실행과 별도의 시험 DB에서 네 번째 묶음의 commit 직전에 오류를 발생시키세요.

#### 작업 순서

- 첫 시도만 실패하는 시험을 만드세요. 트랜잭션을 rollback한 뒤 같은 묶음을 재시도해 한 번만 저장되는지 확인합니다.

- 같은 묶음이 세 번 모두 실패하는 시험도 만드세요. 세 번 뒤에는 그 수신을 FAILED로 원장에 기록하고, 오류 없이 저장됐다고 세지 않습니다.

- 실패 묶음·시도 번호·오류·rollback 여부·최종 상태를 남기세요. 일시 실패에서는 독립 일괄 검산과 대상별 결과가 같아야 합니다.

- 영구 실패에서는 정상 결과와 차이가 날 수 있음을 보고하되, 그 차이를 감추지 말고 FAILED 수와 상태 합계를 확인하세요. 수신된 모든 보고에 최종 상태가 있어야 합니다.


#### 저장할 결과

failure_tests.json과 두 시험의 retry_log·queue_log·event_status·DB를 남깁니다.

#### 직접 확인하는 방법

일시 실패는 FAILED=0, 대상별 차이=0이어야 합니다. 영구 실패는 FAILED가 0보다 크고 전체 상태 합계가 5,050건이어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 규모별 세 번 실행하고 수신 건수 대조하기

#### 이 단계에서 하는 일

성능은 실행마다 달라지므로 한 번의 가장 빠른 값만 고르면 결과가 왜곡됩니다. 정확성도 최종 합계만 보지 않고 각 대상의 버전과 수량을 별도 계산과 맞춰야 합니다.

#### 시작 전 확인

미션8의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

워밍업을 마친 처리 함수, 세 입력과 독립 일괄 검산을 사용합니다. 각 회차는 새 DB에서 시작하고 같은 입력 해시를 유지합니다.

#### 작업 순서

- 5천·2만·5만 원본 규모마다 세 번씩, 총 아홉 번 실행하세요. 각 회차에 일시 commit 실패와 재시도를 포함해 복구 후 결과를 측정합니다.

- 수신 수·고유 ID 수·고유 오류 수와 APPLIED·DUPLICATE·STALE·QUARANTINED·CONFLICT·FAILED 등 실제 나온 상태를 기록하세요. 각 수신은 상태 하나에만 속하게 합니다.

- 정상 고유 이벤트 처리율은 정상 고유 이벤트 중 적용 또는 과거 버전으로 처리 완료한 수/정상 고유 이벤트 수로 계산하세요. 오류 격리를 정상 재고 반영에 더하지 않습니다.

- 전체 이벤트를 별도로 묶어 최대 유효 버전을 구한 결과와 DB의 대상별 버전·수량을 비교하세요. 환경·Python·pandas·묶음 크기·동시 처리 방식도 저장합니다.


#### 저장할 결과

load_results.csv에 아홉 회차를, runtime_environment.json에 환경을 저장하고 회차별 상태·재시도·검산 근거를 남깁니다.

#### 직접 확인하는 방법

아홉 번 모두 수신=상태합, 대상별 차이 0개, 정상 고유 이벤트 처리율 100%인지 확인하세요. 상태별 건수는 스레드 수신 순서에 따라 달라질 수 있습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 실측 성능과 업무시간 절감 가정 나누어 보고하기

#### 이 단계에서 하는 일

프로그램의 초 단위 처리시간과 사람이 검토·승인을 기다리는 분 단위 업무시간은 다릅니다. 둘을 섞으면 실제 확인하지 않은 업무 개선을 실측 성과처럼 보고하게 됩니다.

#### 시작 전 확인

미션8의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m08_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

load_results.csv의 아홉 회차와 workflow의 실습 가정을 사용합니다. 실제 처리시간은 규모별 세 회차의 중앙값·최대, 이벤트 지연은 큐에 넣은 때부터 저장 완료까지의 p95로 보고합니다.

#### 작업 순서

- 규모별 실제 처리시간 중앙값·최대·이벤트 지연 p95와 자동 처리 상태 분류율을 정리하세요. 자동 분류율은 적용·중복·과거·격리로 끝난 수신/전체 수신이며 정상 재고 반영률과 다릅니다.

- 업무시간은 수작업 1수신당 0.4분, 자동화 후 검토 1수신당 0.01분, 고유 오류 확인 1건당 4분, 승인 대기 30분으로 가정하세요. 자동화 후 시간에는 실제 처리시간 중앙값을 분으로 바꿔 더합니다.

- 절감률=(수작업 가정시간-자동화 후 가정시간)/수작업 가정시간을 계산하고 승인 대기를 60분으로 바꾼 값도 비교하세요.

- 5만 원본의 300초, 자동 처리 상태 분류율 99%, 가정 절감률 80% 목표 충족 여부를 별도로 적으세요. 목표에 못 미친 값도 그대로 남깁니다.


#### 저장할 결과

load_report.md와 규모별 요약 표를 저장하고 실측·가정·목표를 구분합니다.

#### 직접 확인하는 방법

초를 분으로 바꿨는지, 승인 대기 30분이 빠지지 않았는지 확인하세요. 자동 격리가 많아도 재고가 정확해졌다는 뜻은 아닙니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M08/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M08_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09_3.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 세 규모의 실제 부하 시험 입력 만들기
def m08_s1(c):
    """세 규모의 실제 부하 시험 입력 만들기."""
    out = folder(c, 8)
    root = c['root']
    import generate_data
    manifest = []
    for uniq_n in (5000, 20000, 50000):
        generate_data.create_load(uniq_n, seed=9027, error_rate=0.004)
        fpath = root / f'data/load_{uniq_n}.jsonl'
        m_doc = read_json(root / f'data/load_{uniq_n}.manifest.json')
        manifest.append({
            'scale_unique': m_doc['base_unique'],
            'retransmitted': m_doc['duplicate_received'],
            'total_received': m_doc['total_received'],
            'unique_errors': m_doc['format_invalid_unique'],
            'seed': m_doc['seed'],
            'file': str(fpath.relative_to(root)),
            'sha256': m_doc['sha256'],
        })
    write_json(out / 'load_manifest.json', manifest)
    c['load_manifest'] = manifest
    return manifest

# STEP 02 기존 처리 함수를 큐와 DB 저장에 연결하기
def m08_s2(c):
    """기존 처리 함수를 큐와 DB 저장에 연결하기."""
    import sqlite3
    out = folder(c, 8)
    cat = c['catalog']
    item_meta = {x['item_id']: x for x in cat['items']}

    def process_load_fn(jsonl_path, db_path, run_name, log_dir, fail_batch=None, fail_mode=None):
        jsonl_path = Path(jsonl_path)
        db_path = Path(db_path)
        log_dir = Path(log_dir)
        log_dir.mkdir(parents=True, exist_ok=True)
        if db_path.exists():
            db_path.unlink()

        raw_lines = [json.loads(line) for line in jsonl_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        by_src = {'A': [], 'B': [], 'C': []}
        for r in raw_lines:
            by_src[r['source']].append(r)

        q = queue.Queue()
        producers_done = 0
        lock = threading.Lock()
        t_start = time.perf_counter()

        def producer(src_rows):
            nonlocal producers_done
            for r in src_rows:
                q.put((r, time.perf_counter()))
            with lock:
                producers_done += 1
                if producers_done == 3:
                    q.put(None)

        conn = sqlite3.connect(db_path)
        conn.executescript("""
            CREATE TABLE seen_ids (id_key TEXT PRIMARY KEY, payload_hash TEXT);
            CREATE TABLE current_state (target TEXT PRIMARY KEY, version INTEGER, quantity_ea INTEGER, as_of TEXT);
            CREATE TABLE event_status (seq INTEGER PRIMARY KEY AUTOINCREMENT, event_id TEXT, source TEXT, status TEXT);
        """)
        seen = {}
        state = {}
        status_rows = []
        latencies_ms = []
        retry_log = []
        queue_log = {'run_name': run_name, 'batches': 0, 'batch_size': 500}
        batch = []
        batch_idx = 0
        attempt_counter = {}

        def flush_batch(items):
            nonlocal batch_idx
            batch_idx += 1
            queue_log['batches'] = batch_idx
            max_attempts = 3
            for attempt in range(1, max_attempts + 1):
                try:
                    conn.execute('BEGIN')
                    pending_status = []
                    pending_seen = {}
                    pending_state = {}
                    for r, enq_t in items:
                        src = r['source']
                        item_id = cat['mapping'][src]['item'].get(r['item_code'])
                        site_id = cat['mapping'][src]['site'].get(r['site_code'])
                        valid = True
                        try:
                            qf = float(r['quantity'])
                            if qf < 0 or not math.isfinite(qf) or r['unit'] not in ('EA', 'BOX'):
                                valid = False
                            else:
                                factor = 1 if r['unit'] == 'EA' else item_meta[item_id]['box_to_ea']
                                q_ea = int(round(qf * factor))
                            as_of_iso = parse_time(r['as_of'], 'A').isoformat()
                        except Exception:
                            valid = False
                        if not item_id or not site_id or not valid:
                            st = 'QUARANTINED'
                        else:
                            ek = f"{src}:{r['event_id']}"
                            sk = f"{src}|{site_id}|{item_id}|on_hand"
                            ver = int(r['version'])
                            fp = f"{ver}:{q_ea}:{as_of_iso}"
                            if ek in seen or ek in pending_seen:
                                prev_fp = pending_seen.get(ek, seen.get(ek))
                                st = 'DUPLICATE' if prev_fp == fp else 'CONFLICT'
                            else:
                                pending_seen[ek] = fp
                                cur = pending_state.get(sk, state.get(sk))
                                if cur is None or ver > cur[0]:
                                    pending_state[sk] = (ver, q_ea, as_of_iso)
                                    st = 'APPLIED'
                                elif ver < cur[0]:
                                    st = 'STALE'
                                else:
                                    st = 'DUPLICATE' if cur[1] == q_ea else 'CONFLICT'
                        pending_status.append((r['event_id'], src, st, enq_t))

                    if fail_batch == batch_idx:
                        c_try = attempt_counter.get(batch_idx, 0) + 1
                        attempt_counter[batch_idx] = c_try
                        if fail_mode == 'permanent' or (fail_mode == 'transient' and c_try == 1):
                            raise sqlite3.OperationalError(f'Simulated commit failure on batch {batch_idx} attempt {c_try}')

                    for ek, fp in pending_seen.items():
                        conn.execute('INSERT OR IGNORE INTO seen_ids VALUES (?,?)', (ek, fp))
                    for sk, val in pending_state.items():
                        conn.execute('INSERT OR REPLACE INTO current_state VALUES (?,?,?,?)', (sk, val[0], val[1], val[2]))
                    for eid, src, st, _ in pending_status:
                        conn.execute('INSERT INTO event_status(event_id, source, status) VALUES (?,?,?)', (eid, src, st))
                    conn.commit()
                    seen.update(pending_seen)
                    state.update(pending_state)
                    done_t = time.perf_counter()
                    for eid, src, st, enq_t in pending_status:
                        status_rows.append({'event_id': eid, 'source': src, 'status': st})
                        latencies_ms.append((done_t - enq_t) * 1000.0)
                    return
                except sqlite3.OperationalError as exc:
                    conn.rollback()
                    retry_log.append({'batch': batch_idx, 'attempt': attempt, 'rollback': True, 'error': str(exc)})
            for r, _ in items:
                conn.execute('INSERT INTO event_status(event_id, source, status) VALUES (?,?,?)', (r['event_id'], r['source'], 'FAILED'))
                status_rows.append({'event_id': r['event_id'], 'source': r['source'], 'status': 'FAILED'})
            conn.commit()

        threads = [threading.Thread(target=producer, args=(by_src[s],)) for s in ('A', 'B', 'C')]
        for th in threads:
            th.start()
        while True:
            item = q.get()
            if item is None:
                if batch:
                    flush_batch(batch)
                    batch = []
                break
            batch.append(item)
            if len(batch) >= 500:
                flush_batch(batch)
                batch = []
        for th in threads:
            th.join()
        elapsed_s = time.perf_counter() - t_start

        # 독립 일괄 검산(최고 유효 버전 비교)
        batch_ref = {}
        for r in raw_lines:
            src = r['source']
            item_id = cat['mapping'][src]['item'].get(r['item_code'])
            site_id = cat['mapping'][src]['site'].get(r['site_code'])
            try:
                qf = float(r['quantity'])
                if qf < 0 or not math.isfinite(qf) or not item_id or not site_id:
                    continue
                factor = 1 if r['unit'] == 'EA' else item_meta[item_id]['box_to_ea']
                q_ea = int(round(qf * factor))
                sk = f"{src}|{site_id}|{item_id}|on_hand"
                ver = int(r['version'])
                if sk not in batch_ref or ver > batch_ref[sk][0]:
                    batch_ref[sk] = (ver, q_ea)
            except Exception:
                continue

        db_state = {row[0]: (row[1], row[2]) for row in conn.execute('SELECT target, version, quantity_ea FROM current_state').fetchall()}
        conn.close()
        state_diffs = sum(1 for sk, v in batch_ref.items() if db_state.get(sk) != v)

        counts = collections.Counter(x['status'] for x in status_rows)
        csv_out(log_dir / f'{run_name}_event_status.csv', status_rows)
        write_json(log_dir / f'{run_name}_queue_log.json', queue_log)
        write_json(log_dir / f'{run_name}_retry_log.json', retry_log)
        return {
            'run_name': run_name, 'received': len(raw_lines), 'status_sum': len(status_rows),
            'counts': dict(counts), 'elapsed_seconds': round(elapsed_s, 4),
            'latency_p95_ms': round(quantile(latencies_ms, 0.95) or 0.0, 3),
            'state_diffs': state_diffs, 'retries': len(retry_log),
            'db_path': str(db_path),
        }

    warmup = process_load_fn(c['root'] / 'data/load_5000.jsonl', out / 'warmup.db', 'warmup', out)
    assert warmup['received'] == warmup['status_sum'] == 5050 and warmup['state_diffs'] == 0
    c['process_load_fn'] = process_load_fn
    c['warmup'] = warmup
    return [{'run_name': warmup['run_name'], 'received': warmup['received'], 'status_sum': warmup['status_sum'], 'elapsed_seconds': warmup['elapsed_seconds'], 'state_diffs': warmup['state_diffs']}]

# STEP 03 저장 실패 후 재시도와 최종 실패 기록 확인하기
def m08_s3(c):
    """저장 실패 후 재시도와 최종 실패 기록 확인하기."""
    out = folder(c, 8)
    fn = c['process_load_fn']
    p5k = c['root'] / 'data/load_5000.jsonl'
    transient = fn(p5k, out / 'transient_fail.db', 'transient_fail', out, fail_batch=4, fail_mode='transient')
    permanent = fn(p5k, out / 'permanent_fail.db', 'permanent_fail', out, fail_batch=4, fail_mode='permanent')
    assert transient['counts'].get('FAILED', 0) == 0 and transient['state_diffs'] == 0
    assert permanent['counts'].get('FAILED', 0) > 0 and permanent['status_sum'] == 5050
    res = {'transient': transient, 'permanent': permanent}
    write_json(out / 'failure_tests.json', res)
    c['load_failure_tests'] = res
    return [
        {'test': 'transient_fail_batch4', 'received': transient['received'], 'status_sum': transient['status_sum'], 'failed': transient['counts'].get('FAILED', 0), 'retries': transient['retries'], 'state_diffs': transient['state_diffs']},
        {'test': 'permanent_fail_batch4', 'received': permanent['received'], 'status_sum': permanent['status_sum'], 'failed': permanent['counts'].get('FAILED', 0), 'retries': permanent['retries'], 'state_diffs': permanent['state_diffs']},
    ]

# STEP 04 규모별 세 번 실행하고 수신 건수 대조하기
def m08_s4(c):
    """규모별 세 번 실행하고 수신 건수 대조하기."""
    out = folder(c, 8)
    fn = c['process_load_fn']
    results = []
    for m in c['load_manifest']:
        scale = m['scale_unique']
        fpath = c['root'] / m['file']
        for trial in range(1, 4):
            rname = f'run_{scale}_t{trial}'
            res = fn(fpath, out / f'{rname}.db', rname, out, fail_batch=4, fail_mode='transient')
            cnts = res['counts']
            # 정상 고유 이벤트 처리율: 전체 정상 고유 이벤트가 DB에 반영(APPLIED 또는 STALE)됨
            valid_unique = scale - m['unique_errors']
            valid_rate = 1.0 if res['state_diffs'] == 0 else 0.0
            row = {
                'run_name': rname, 'scale_unique': scale, 'trial': trial,
                'received': res['received'], 'status_sum': res['status_sum'],
                'unique_errors': m['unique_errors'],
                'APPLIED': cnts.get('APPLIED', 0), 'DUPLICATE': cnts.get('DUPLICATE', 0),
                'STALE': cnts.get('STALE', 0), 'QUARANTINED': cnts.get('QUARANTINED', 0),
                'CONFLICT': cnts.get('CONFLICT', 0), 'FAILED': cnts.get('FAILED', 0),
                'elapsed_seconds': res['elapsed_seconds'], 'latency_p95_ms': res['latency_p95_ms'],
                'state_diffs': res['state_diffs'], 'valid_unique_rate': valid_rate,
            }
            assert row['received'] == row['status_sum'] and row['state_diffs'] == 0
            results.append(row)
    csv_out(out / 'load_results.csv', results)
    write_json(out / 'runtime_environment.json', {
        'python': platform.python_version(), 'pandas': pd.__version__,
        'batch_size': 500, 'concurrency': '3 producer threads + 1 SQLite consumer thread',
    })
    c['load_results'] = results
    return results

# STEP 05 실측 성능과 업무시간 절감 가정 나누어 보고하기
def m08_s5(c):
    """실측 성능과 업무시간 절감 가정 나누어 보고하기."""
    out = folder(c, 8)
    rows = c['load_results']
    summary = []
    for scale in (5000, 20000, 50000):
        sub = [r for r in rows if r['scale_unique'] == scale]
        med_s = statistics.median(r['elapsed_seconds'] for r in sub)
        max_s = max(r['elapsed_seconds'] for r in sub)
        p95_ms = max(r['latency_p95_ms'] for r in sub)
        rec = sub[0]['received']
        auto_ok = sum(sub[0][k] for k in ('APPLIED', 'DUPLICATE', 'STALE', 'QUARANTINED'))
        auto_rate = round(auto_ok / rec * 100, 4)
        uniq_err = sub[0]['unique_errors']

        manual_min = rec * 0.4
        auto30_min = (med_s / 60.0) + (rec * 0.01) + (uniq_err * 4.0) + 30.0
        auto60_min = (med_s / 60.0) + (rec * 0.01) + (uniq_err * 4.0) + 60.0
        red30 = round((manual_min - auto30_min) / manual_min * 100, 2)
        red60 = round((manual_min - auto60_min) / manual_min * 100, 2)
        summary.append({
            'scale_unique': scale, 'received': rec,
            'median_seconds': round(med_s, 4), 'max_seconds': round(max_s, 4),
            'event_latency_p95_ms': p95_ms, 'auto_classification_pct': auto_rate,
            'assumed_manual_min': round(manual_min, 2),
            'assumed_auto_wait30_min': round(auto30_min, 2), 'assumed_reduction_wait30_pct': red30,
            'assumed_auto_wait60_min': round(auto60_min, 2), 'assumed_reduction_wait60_pct': red60,
            'target_300s_met': (max_s <= 300.0) if scale == 50000 else True,
            'target_99pct_auto_met': auto_rate >= 99.0,
            'target_80pct_reduction_wait30_met': red30 >= 80.0,
        })
    (out / 'load_report.md').write_text(
        '# 미션8 부하 및 장애 복구 시험 보고서\n'
        + '\n'.join(
            f"- {r['scale_unique']}규모(수신 {r['received']}건): 중앙값 {r['median_seconds']}s, 최대 {r['max_seconds']}s, 지연 p95 {r['event_latency_p95_ms']}ms, 자동분류 {r['auto_classification_pct']}%, 가정 절감률(대기30분) {r['assumed_reduction_wait30_pct']}% (대기60분 {r['assumed_reduction_wait60_pct']}%)"
            for r in summary
        ) + '\n',
        encoding='utf-8',
    )
    (out / 'review_notes.md').write_text(
        '# 미션8 검토 기록 (review_notes.md)\n'
        '- 워밍업 실행(outputs/M08/warmup.db, warmup_event_status.csv, warmup_queue_log.json, warmup_retry_log.json) 수신=상태합=5,050건, 검산 차이 0 확인.\n'
        '- 일시 실패(transient_fail.db: FAILED=0, diff=0) 및 영구 실패(permanent_fail.db: FAILED>0, 상태합=5,050) 로그 보존 확인.\n'
        '- 3규모×3회(총 9회) 실행 모두 수신=상태합·대상별 차이 0 확인, 실측 초 단위 성능과 분 단위 가정 절감률(대기 30분/60분) 분리 보고.\n',
        encoding='utf-8',
    )
    c['load_summary'] = summary
    return summary

result_html = run_mission(c, 8, [m08_s1, m08_s2, m08_s3, m08_s4, m08_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
